In [ ]:

from pyspark.sql import functions as F

from atlas.common.config.loader import get_settings
from atlas.common.paths.loader import get_paths
from atlas.common.spark.session import get_spark_session

settings = get_settings("configs/base.yaml", "configs/local.yaml", "pyproject.toml")

In [ ]:
spark = get_spark_session(settings.spark, settings.storage, settings.application.name)

In [ ]:
paths = get_paths(settings)

In [ ]:
bronze_customer_path = paths.bronze_path("customer/cdc/customers/job")

In [ ]:
customer_bronze_data = spark.read.format("parquet").load(bronze_customer_path)

In [ ]:

customer_bronze_data.select(F.col("raw_key"), F.col("raw_value")).show(vertical=True, truncate=False, n=1)

In [ ]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

customer_record_schema = StructType([
    StructField("customer_id", LongType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("date_of_birth", LongType(), True),
    StructField("status", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("created_at", StringType(), False),
    StructField("updated_at", StringType(), False),
])

In [ ]:
customer_source_schema = StructType([
    StructField("version", StringType(), False),
    StructField("connector", StringType(), False),
    StructField("name", StringType(), False),
    StructField("ts_ms", LongType(), False),
    StructField("snapshot", StringType(), True),
    StructField("db", StringType(), False),
    StructField("sequence", StringType(), True),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
    StructField("schema", StringType(), False),
    StructField("table", StringType(), False),
    StructField("txId", LongType(), True),
    StructField("lsn", LongType(), True),
    StructField("xmin", LongType(), True),
])

In [ ]:
customer_payload_schema = StructType([
    StructField("before", customer_record_schema, True),
    StructField("after", customer_record_schema, True),
    StructField("source", customer_source_schema, False),
    StructField("op", StringType(), False),
    StructField("ts_ms", LongType(), False),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
])

In [ ]:
customer_debezium_schema = StructType([
    StructField("payload", customer_payload_schema, True),
])

In [ ]:
customer_parsed_data = customer_bronze_data.withColumn("debezium", F.from_json(F.col("raw_value")
                                                                               , customer_debezium_schema))

In [ ]:
customer_parsed_data.select(
    "debezium.payload.before",
    "debezium.payload.after",
    "debezium.payload.op",
    "debezium.payload.source.lsn",
    "kafka_partition",
    "kafka_offset",
).show(1, truncate=False, vertical=True)

In [ ]:
customer_cdc_data = customer_parsed_data.withColumn("customer",
                                                    F.when(
                                                        F.col("debezium.payload.op") == "d",
                                                        F.col("debezium.payload.before"),

                                                    ).otherwise(
                                                        F.col("debezium.payload.after")
                                                    )
                                                    )

In [ ]:
customer_cdc_data_normalized = customer_cdc_data.select(
    F.col("customer.customer_id").alias("customer_id"),
    F.col("customer.first_name").alias("first_name"),
    F.col("customer.last_name").alias("last_name"),
    F.col("customer.email").alias("email"),
    F.col("customer.phone_number").alias("phone_number"),
    F.date_add(
        F.lit("1970-01-01").cast("date"),
        F.col("customer.date_of_birth").cast("int")
    ).alias("date_of_birth"),
    F.col("customer.status").alias("status"),
    F.col("customer.segment").alias("segment"),
    F.try_to_timestamp(F.col("customer.created_at")).alias("created_at"),
    F.try_to_timestamp(F.col("customer.updated_at")).alias("updated_at"),
    F.timestamp_millis(F.col("debezium.payload.ts_ms")).alias("cdc_timestamp"),
    F.timestamp_millis(F.col("debezium.payload.source.ts_ms")).alias("source_timestamp"),
    F.col("debezium.payload.op").alias("cdc_operation"),
    F.col("debezium.payload.source.lsn").alias("source_lsn"),
    F.col("debezium.payload.source.txId").alias("source_tx_id"),
    F.col("kafka_topic").alias("kafka_topic"),
    F.col("kafka_partition").alias("kafka_partition"),
    F.col("kafka_offset").alias("kafka_offset"),
    F.col("kafka_timestamp").alias("kafka_timestamp"),
    F.col("is_tombstone").alias("is_tombstone"),
    F.col("ingested_at").alias("ingested_at"),
)

In [ ]:
customer_non_tombstone_data = customer_cdc_data_normalized.filter(
    ~F.col("is_tombstone")
)

In [ ]:
customer_filter_condition = (
    F.array(
        F.when(F.col("customer_id").isNull(), F.lit("MISSING_CUSTOMER_ID")),
        F.when((F.col("first_name").isNull()| (F.trim(F.col("first_name")) == "")), F.lit("MISSING_FIRST_NAME")),
        F.when((F.col("last_name").isNull() |(F.trim(F.col("last_name")) == "")), F.lit("MISSING_LAST_NAME")),
        F.when((F.col("email").isNull() & F.col("phone_number").isNull()), F.lit("MISSING_CONTACT_INFO")),
        F.when(F.col("date_of_birth") > F.current_date(), F.lit("FUTURE_DATE_OF_BIRTH")),
        F.when(~F.col("status").isin(["ACTIVE","INACTIVE","SUSPENDED"]), F.lit("INVALID_STATUS")),
        F.when(~F.col("segment").isin(["STANDARD","GOLD","PREMIUM"]), F.lit("INVALID_SEGMENT"))
))

In [ ]:
customer_filtered_data = customer_non_tombstone_data.withColumn("dq_errors", F.array_compact(customer_filter_condition))

In [ ]:
customer_filtered_data.select(["customer_id","first_name","last_name","email","phone_number"
                                  ,"cdc_operation","is_tombstone","dq_errors" ]).show(truncate=False)

In [ ]:
(customer_filtered_data.select(["customer_id","first_name","last_name","email",
                               "phone_number","cdc_operation","is_tombstone","dq_errors" ])
 .show(truncate=False))

In [ ]:
customer_valid_data = customer_filtered_data.filter(F.size(F.col("dq_errors")) ==0)
customer_quarantine_data = customer_filtered_data.filter(F.size(F.col("dq_errors")) >0)

In [ ]:
customer_quarantine_data = (customer_quarantine_data.withColumn("dq_error_count", F.size(F.col("dq_errors")))
                            .withColumn("quarantined_at", F.current_timestamp()))

In [ ]:
# Persist DQ-invalid customer records for investigation
silver_customer_quarantine_path = paths.silver_path(
    "customer/cdc/customers/quarantine/notebook"
)


In [ ]:
customer_quarantine_data.show(truncate=False, vertical=True)

In [ ]:
customer_valid_data = customer_valid_data.drop("dq_errors")

In [ ]:
customer_deduplicated_data = customer_valid_data.drop_duplicates(["kafka_topic","kafka_partition", "kafka_offset"])

In [ ]:
customer_deduplicated_data.select(
    "customer_id",
    "cdc_operation",
    "source_tx_id",
    "source_lsn",
    "kafka_partition",
    "kafka_offset",
).orderBy(
    "kafka_partition",
    "kafka_offset",
).show(truncate=False)


In [ ]:
customer_deduplicated_data.count()

In [ ]:
customer_incoming_ambiguous_keys = (customer_deduplicated_data.groupBy(['customer_id', 'source_lsn'])
                                    .agg(F.countDistinct("kafka_partition").alias("distinct_partition") )
                                    .filter(F.col("distinct_partition")>=2))

In [ ]:
customer_incoming_ambiguous_events = customer_deduplicated_data.join(customer_incoming_ambiguous_keys,
                                                                     ["customer_id","source_lsn"],"left_semi")
customer_incoming_ambiguous_events = (
    customer_incoming_ambiguous_events
    .withColumn("cdc_status", F.lit("AMBIGUOUS_ORDERING"))
    .withColumn("persisted_source_lsn", F.lit(None).cast("long"))
    .withColumn("persisted_kafka_partition", F.lit(None).cast("int"))
    .withColumn("persisted_kafka_offset", F.lit(None).cast("long"))
    .withColumn("rejected_at", F.current_timestamp())
)

In [ ]:
customer_incoming_orderable_data = customer_deduplicated_data.join(
    customer_incoming_ambiguous_keys,
    ["customer_id", "source_lsn"],
    "left_anti"
)

In [ ]:
# S3 - CDC ordering, replay protection and persistent history

from delta.tables import DeltaTable
from pyspark.sql import Window


In [ ]:
# Persistent Delta table containing observed CDC event history

silver_customer_history_path = paths.silver_path(
    "customer/cdc/customers/cdc_history/notebook"
)
# Check history before processing this run

customer_history_exists = DeltaTable.isDeltaTable(
    spark,
    silver_customer_history_path
)

In [ ]:
if customer_history_exists:

    # Read PREVIOUS accepted history
    customer_cdc_history_table = DeltaTable.forPath(
        spark,
        silver_customer_history_path
    )

    customer_cdc_history_data = customer_cdc_history_table.toDF()

    # Latest accepted event for each customer
    customer_cdc_latest_window = Window.partitionBy(
        "customer_id"
    ).orderBy(
        F.col("source_lsn").desc(),
        F.col("kafka_offset").desc()
    )

    customer_cdc_latest = (
        customer_cdc_history_data
        .withColumn(
            "row_number",
            F.row_number().over(customer_cdc_latest_window)
        )
        .filter(F.col("row_number") == 1)
        .drop("row_number")
    )

    # Compare this run's incoming events against PREVIOUS history
    customer_cdc_comparison = (
       customer_incoming_orderable_data.alias("s")
        .join(
            customer_cdc_latest.alias("t"),
            F.col("s.customer_id") == F.col("t.customer_id"),
            "left"
        )
    )

    # Classify source ordering
    customer_cdc_classified = customer_cdc_comparison.withColumn(
        "cdc_status",

        F.when(
            F.col("t.customer_id").isNull(),
            F.lit("NEW")
        )
        .when(
            F.col("s.source_lsn") > F.col("t.source_lsn"),
            F.lit("NEWER")
        )
        .when(
            F.col("s.source_lsn") < F.col("t.source_lsn"),
            F.lit("STALE")
        )
        .when(
            F.col("s.kafka_partition") != F.col("t.kafka_partition"),
            F.lit("AMBIGUOUS_ORDERING")
        )
        .when(
            F.col("s.kafka_offset") > F.col("t.kafka_offset"),
            F.lit("NEWER")
        )
        .otherwise(
            F.lit("STALE")
        )
    )

    # Only accepted incoming events
    customer_cdc_accepted_events = (
        customer_cdc_classified
        .filter(F.col("cdc_status").isin("NEW", "NEWER"))
        .select("s.*")
    )

    # Keep these separately for later monitoring/quarantine
    customer_cdc_rejected_events = (
    customer_cdc_classified
    .filter(
        F.col("cdc_status").isin(
            "STALE",
            "AMBIGUOUS_ORDERING"
        )
    )
    .select(
        # Keep incoming event once
        "s.*",
        # Why we rejected it
        "cdc_status",
        # Previous accepted position it conflicted with
        F.col("t.source_lsn").alias("persisted_source_lsn"),
        F.col("t.kafka_partition").alias(
            "persisted_kafka_partition"
        ),
        F.col("t.kafka_offset").alias(
            "persisted_kafka_offset"
        )
    )
    .withColumn(
        "rejected_at",
        F.current_timestamp()
    )
)
else:
    # FIRST RUN:
    # There is no previous history to compare against.
    customer_cdc_accepted_events = customer_incoming_orderable_data
    customer_cdc_rejected_events = customer_incoming_ambiguous_events

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


def merge_cdc_events(
    spark: SparkSession,
    data: DataFrame,
    target_path: str,
) -> None:
    """Persist CDC events idempotently using Kafka record identity."""

    if not DeltaTable.isDeltaTable(spark, target_path):
        (
            data.write
            .format("delta")
            .save(target_path)
        )
        return

    target_table = DeltaTable.forPath(
        spark,
        target_path
    )

    event_identity_condition = (
        (F.col("t.kafka_topic") == F.col("s.kafka_topic"))
        & (F.col("t.kafka_partition") == F.col("s.kafka_partition"))
        & (F.col("t.kafka_offset") == F.col("s.kafka_offset"))
    )

    (
        target_table.alias("t")
        .merge(
            data.alias("s"),
            event_identity_condition
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [ ]:
silver_customer_cdc_rejected_path = paths.silver_path(
    "customer/cdc/customers/rejected/notebook"
    )
silver_customer_quarantine_path = paths.silver_path(
    "customer/cdc/customers/quarantine/notebook"
)

In [ ]:
# DQ quarantine
merge_cdc_events(
    spark,
    customer_quarantine_data,
    silver_customer_quarantine_path,
)

# Accepted CDC history
merge_cdc_events(
    spark,
    customer_cdc_accepted_events,
    silver_customer_history_path,
)


# Persist CDC ordering rejections
if customer_cdc_rejected_events is not None:
    customer_cdc_rejected_events = (
    customer_cdc_rejected_events
    .unionByName(customer_incoming_ambiguous_events)
)
    merge_cdc_events(
        spark,
        customer_cdc_rejected_events,
        silver_customer_cdc_rejected_path,
    )

In [ ]:
print(customer_cdc_rejected_events)